# TimeDataModel — Quickstart

A tour of `timedatamodel`. This notebook walks through:

1. Building a `TimeSeries` from pandas / Polars / lists / NumPy / PyArrow
2. Round-tripping metadata through a `TimeSeriesDescriptor`
3. The four `DataShape` variants (`SIMPLE`, `VERSIONED`, `CORRECTED`, `AUDIT`)
4. Unit conversion with pint and `validate_unit`
5. Coverage bars for missing data

The only required dependency is `polars`. Pandas, NumPy, PyArrow, and pint are all optional extras used below — install via `pip install timedatamodel[all]` to follow along.

In [1]:
import pandas as pd
import polars as pl

from timedatamodel import (
    DataShape,
    DataType,
    Frequency,
    TimeSeries,
    TimeSeriesDescriptor,
    TimeSeriesType,
)

## 1. Build from pandas

The canonical construction path. A `valid_time` column (or index) plus a `value` column gives a `SIMPLE`-shape series. `name` is the only required metadata field.

In [2]:
df = pd.DataFrame({
    "valid_time": pd.date_range("2024-01-01", periods=24, freq="h", tz="UTC"),
    "value": [100.0 + i * 2.5 for i in range(24)],
})

ts = TimeSeries.from_pandas(
    df,
    name="wind_power",
    unit="MW",
    frequency=Frequency.PT1H,
    data_type=DataType.FORECAST,
    description="Hourly wind power forecast",
)
ts

Metric,wind_power
Shape,SIMPLE
Rows,24
Frequency,PT1H
Timezone,UTC
Unit,MW
Data type,FORECAST
Description,Hourly wind power forecast
valid_time,wind_power
2024-01-01 00:00,100.0
2024-01-01 01:00,102.5


## 2. Descriptor round-trip

`TimeSeriesDescriptor` is a frozen, data-free view of the metadata — useful for registering or cataloging series structure *before* data arrives. It shares every semantic field with `TimeSeries`, so round-trip is lossless.

In [3]:
desc = ts.to_descriptor()
desc

TimeSeriesDescriptor(name='wind_power', unit='MW', data_type=<DataType.FORECAST: 'FORECAST'>, timeseries_type=<TimeSeriesType.FLAT: 'FLAT'>, frequency=<Frequency.PT1H: 'PT1H'>, timezone='UTC', description='Hourly wind power forecast')

In [4]:
# Rehydrate with a different DataFrame — one descriptor can serve many payloads.
ts2 = TimeSeries.from_descriptor(desc, ts.df)
assert ts2.name == ts.name
assert ts2.unit == ts.unit
assert ts2.frequency is ts.frequency
assert ts2.timezone == ts.timezone
ts2

Metric,wind_power
Shape,SIMPLE
Rows,24
Frequency,PT1H
Timezone,UTC
Unit,MW
Data type,FORECAST
Description,Hourly wind power forecast
valid_time,wind_power
2024-01-01 00:00,100.0
2024-01-01 01:00,102.5


## 3. Other input formats

The same metadata kwargs apply to every factory. Pick whichever format your data already lives in; the internal storage is always a timezone-aware Polars `DataFrame`.

In [5]:
df_pl = pl.DataFrame({
    "valid_time": pl.Series(pd.date_range("2024-02-01", periods=6, freq="h", tz="UTC")).cast(pl.Datetime("us", "UTC")),
    "value": [10.0, 11.0, 12.5, 11.8, 13.0, 14.2],
})
TimeSeries.from_polars(df_pl, name="solar_power", unit="MW")

Metric,solar_power
Shape,SIMPLE
Rows,6
Timezone,UTC
Unit,MW
valid_time,solar_power
2024-02-01 00:00,10.0
2024-02-01 01:00,11.0
2024-02-01 02:00,12.5


In [6]:
cols = ts.to_list()            # column-oriented dict
TimeSeries.from_list(cols, name=ts.name, unit=ts.unit, frequency=ts.frequency)

Metric,wind_power
Shape,SIMPLE
Rows,24
Frequency,PT1H
Timezone,UTC
Unit,MW
valid_time,wind_power
2024-01-01 00:00,100.0
2024-01-01 01:00,102.5
2024-01-01 02:00,105.0
…,…


In [7]:
arr = ts.to_numpy()            # dict[str, np.ndarray] — requires numpy
TimeSeries.from_numpy(arr, name=ts.name, unit=ts.unit, frequency=ts.frequency)

Metric,wind_power
Shape,SIMPLE
Rows,24
Frequency,PT1H
Timezone,UTC
Unit,MW
valid_time,wind_power
2024-01-01 00:00,100.0
2024-01-01 01:00,102.5
2024-01-01 02:00,105.0
…,…


In [8]:
tbl = ts.to_pyarrow()          # pa.Table — requires pyarrow
TimeSeries.from_pyarrow(tbl, name=ts.name, unit=ts.unit, frequency=ts.frequency)

Metric,wind_power
Shape,SIMPLE
Rows,24
Frequency,PT1H
Timezone,UTC
Unit,MW
valid_time,wind_power
2024-01-01 00:00,100.0
2024-01-01 01:00,102.5
2024-01-01 02:00,105.0
…,…


## 4. Data shapes

`DataShape` is inferred from the column names:

| Shape       | Columns                                              | Use case                                  |
|-------------|------------------------------------------------------|-------------------------------------------|
| `SIMPLE`    | `valid_time`, `value`                                | Standard time series                      |
| `VERSIONED` | `knowledge_time`, `valid_time`, `value`              | Bi-temporal (when was each value known?)  |
| `CORRECTED` | `valid_time`, `change_time`, `value`                 | Revision history of a single timeline     |
| `AUDIT`     | `knowledge_time`, `change_time`, `valid_time`, `value` | Full audit trail                        |


In [9]:
times = pd.date_range("2024-03-01", periods=4, freq="h", tz="UTC")
df_v = pd.DataFrame({
    "knowledge_time": times,
    "valid_time":     times,
    "value":          [5.1, 5.3, 5.0, 5.4],
})
ts_v = TimeSeries.from_pandas(df_v, name="load", unit="MW")
print("shape:", ts_v.shape)
ts_v

shape: DataShape.VERSIONED


TimeSeries
┌────────────────────────────────────────────┐
│  Metric:           load                    │
│  Shape:            VERSIONED               │
│  Rows:             4                       │
│  Timezone:         UTC                     │
│  Unit:             MW                      │
├────────────────────────────────────────────┤
│                                      load  │
│  2024-03-01 00:00, 2024-03-01 00:00   5.1  │
│  2024-03-01 01:00, 2024-03-01 01:00   5.3  │
│  2024-03-01 02:00, 2024-03-01 02:00   5.0  │
│  2024-03-01 03:00, 2024-03-01 03:00   5.4  │
└────────────────────────────────────────────┘

In [10]:
# Round-trip to pandas restores the conventional MultiIndex for each shape.
ts_v.to_pandas()

,,value
knowledge_time,valid_time,
2024-03-01 00:00:00+00:00,2024-03-01 00:00:00+00:00,5.1
2024-03-01 01:00:00+00:00,2024-03-01 01:00:00+00:00,5.3
2024-03-01 02:00:00+00:00,2024-03-01 02:00:00+00:00,5.0
2024-03-01 03:00:00+00:00,2024-03-01 03:00:00+00:00,5.4


## 5. Units — conversion and validation

Both features require the `[pint]` extra. `convert_unit` rescales the values and updates the `unit` metadata; `validate_unit` simply checks whether a string parses as a pint unit.

In [11]:
ts_kw = ts.convert_unit("kW")
print("original unit :", ts.unit,    "first value:", ts.df["value"][0])
print("converted unit:", ts_kw.unit, "first value:", ts_kw.df["value"][0])

original unit : MW first value: 100.0
converted unit: kW first value: 100000.0


In [12]:
from timedatamodel.units import validate_unit

print("'MW'        →", validate_unit("MW"))
print("'kg*m/s**2' →", validate_unit("kg*m/s**2"))
print("'garbage'   →", validate_unit("garbage"))

'MW'        → True
'kg*m/s**2' → True
'garbage'   → False


## 6. Coverage bar

For series with gaps, `coverage_bar()` returns a binned view of value presence. In Jupyter it renders as an SVG; in a terminal it uses Unicode blocks.

In [13]:
import numpy as np

rng = np.random.default_rng(0)
values = [float(i) if rng.random() > 0.25 else None for i in range(48)]
df_missing = pd.DataFrame({
    "valid_time": pd.date_range("2024-04-01", periods=48, freq="h", tz="UTC"),
    "value":      values,
})
ts_missing = TimeSeries.from_pandas(df_missing, name="meter_reading", unit="kWh", frequency=Frequency.PT1H)
ts_missing.coverage_bar()

meter_reading  ██░░███████░█░█░████░░██████████░█████████████░█
               2024-04-01 00:00                2024-04-02 23:00